NAME: SAIJYOTI PANDA

SRN: PES2UG23CS512

SEC: H


In [18]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-groq \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    crewai \
    crewai-tools \
    deepeval \
    groq \
    litellm \
    pydantic

print("All packages installed successfully!")

All packages installed successfully!


## Part 1: Knowledge Base

**Topic: James Webb Space Telescope (JWST)**

**Why chosen:** JWST is an ideal RAG knowledge base because it contains
many distinct, verifiable numerical facts that stress-test
both Faithfulness and Answer Relevancy. It also lends itself naturally to
adversarial questions — topics like other space programs or planetary science that are absent from the corpus.


In [19]:
import os, warnings
warnings.filterwarnings("ignore")

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

os.environ["OPENAI_API_KEY"]  = GROQ_API_KEY
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"
os.environ["GROQ_API_KEY"]    = GROQ_API_KEY

print("Groq API key configured correctly")

Groq API key configured correctly


In [20]:
knowledge_base_text = """
The James Webb Space Telescope (JWST) is the largest and most powerful space
telescope ever built. It was launched on December 25, 2021, aboard an Ariane 5
rocket from the Guiana Space Centre in Kourou, French Guiana. The telescope is a
joint project of NASA, the European Space Agency (ESA), and the Canadian Space
Agency (CSA). Its total development cost was approximately 10 billion US dollars,
making it one of the most expensive scientific instruments in history.

JWST operates at the second Sun-Earth Lagrange point (L2), located approximately
1.5 million kilometres (about 1 million miles) from Earth in the direction
opposite the Sun. This location keeps the telescope in a stable thermal
environment and allows its sunshield to simultaneously block light and heat from
the Sun, Earth, and Moon.

The telescope's primary mirror is 6.5 metres (21.3 feet) in diameter and is
composed of 18 hexagonal segments coated with a thin layer of gold. The segments
are made of beryllium, a light yet extremely rigid metal. The total collecting
area of the mirror is 25.4 square metres, compared to 4.5 square metres for the
Hubble Space Telescope. The larger mirror allows JWST to collect about six times
more light than Hubble.

JWST is an infrared telescope, designed to observe the universe primarily in
infrared wavelengths (0.6 to 28 micrometres). This enables it to see through dust
clouds that block visible light and to observe very distant objects whose light
has been redshifted into the infrared by the expansion of the universe.

The telescope carries four main science instruments:
1. NIRCam (Near Infrared Camera) — the primary imager.
2. NIRSpec (Near Infrared Spectrograph) — can observe up to 100 objects simultaneously.
3. MIRI (Mid-Infrared Instrument) — covers mid-infrared wavelengths and requires
   active cooling to 7 Kelvin (−266 °C), the coldest instrument on the telescope.
4. FGS/NIRISS (Fine Guidance Sensor / Near Infrared Imager and Slitless
   Spectrograph) — guides pointing and performs slitless spectroscopy.

JWST's five-layer sunshield is roughly the size of a tennis court
(21 m × 14 m) and is made of a material called Kapton coated with aluminium and
doped silicon. The sunshield keeps the telescope's instruments at approximately
−233 °C (40 Kelvin), cold enough to detect the faint infrared glow of distant
galaxies without being overwhelmed by the telescope's own heat.

The telescope was designed for a minimum science mission of 10 years. However,
because the Ariane 5 launch was so precise, JWST consumed far less fuel during
its trajectory corrections than planned, giving it an operational lifetime
estimated at more than 20 years.

NASA released JWST's first full-colour images and spectroscopic data on
July 12, 2022. The initial release included the deepest and sharpest infrared
image of the distant universe ever taken, showing the galaxy cluster SMACS 0723.
JWST is designed to observe the first galaxies that formed after the Big Bang,
more than 13.5 billion years ago.

One of JWST's early scientific breakthroughs was the detection of carbon dioxide
(CO2) in the atmosphere of the exoplanet WASP-39b, a hot gas giant approximately
700 light-years from Earth. This marked the first clear detection of CO2 in the
atmosphere of a planet outside our solar system and demonstrated JWST's power
for exoplanet atmospheric characterisation.

JWST has also observed Jupiter in unprecedented detail, capturing images that
reveal the planet's auroras, rings, and two of its moons — Amalthea and Adrastea.
In addition, the telescope detected water vapour plumes erupting from the surface
of Saturn's moon Enceladus, one of the most promising targets in the search for
extraterrestrial life.

JWST represents a successor to both the Hubble Space Telescope and the Spitzer
Space Telescope, combining the deep-field capability of Hubble with the infrared
vision of Spitzer at far greater sensitivity.
"""

print(f"Knowledge base word count: {len(knowledge_base_text.split())} words")
print("Knowledge base loaded.")

Knowledge base word count: 620 words
Knowledge base loaded.


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
    length_function=len,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_text(knowledge_base_text)
docs   = [Document(page_content=chunk) for chunk in chunks]

print(f"Number of chunks : {len(docs)}")
print(f"Avg chunk size   : {sum(len(d.page_content) for d in docs) // len(docs)} chars")
print("\nSample chunk:")
print("─" * 60)
print(docs[2].page_content)

Number of chunks : 16
Avg chunk size   : 244 chars

Sample chunk:
────────────────────────────────────────────────────────────
JWST operates at the second Sun-Earth Lagrange point (L2), located approximately
1.5 million kilometres (about 1 million miles) from Earth in the direction
opposite the Sun. This location keeps the telescope in a stable thermal
environment and allows its sunshield to simultaneously block light and heat from
the Sun, Earth, and Moon.


In [22]:
print("Loading embedding model (first run may take ~1 min to download)...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})


print(f"FAISS vector store built with {vectorstore.index.ntotal} vectors.")

test_docs = retriever.invoke("What is the primary mirror diameter of JWST?")
print(f"\nRetrieval test — top chunk:\n{test_docs[0].page_content[:200]}...")

Loading embedding model (first run may take ~1 min to download)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector store built with 16 vectors.

Retrieval test — top chunk:
Hubble Space Telescope. The larger mirror allows JWST to collect about six times
more light than Hubble....


## Part 2: RAG Agent

The RAG agent is a CrewAI Agent equipped with
rag_search. When given a question:

1. It calls rag_search(query) which hits the FAISS retriever and returns
   the top-3 most relevant chunks.
2. It generates a factual answer grounded strictly in those chunks.
3. It returns a JSON string containing question, answer, and context
   

The LLM backend is `llama-3.3-70b-versatile` (temperature=0.1 for
consistency).

In [23]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

crewai_llm = LLM(
    model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.1,
)

# ── RAG Tool ───────────────────────────────────────────────────────────────
@tool("rag_search")
def rag_search(query: str) -> str:
    """
    Search the JWST knowledge base and return the top-3 most relevant
    document chunks for the given query.
    """
    retrieved = retriever.invoke(query)
    context   = "\n\n---\n\n".join(d.page_content for d in retrieved)
    return context

# ── RAG Agent ──────────────────────────────────────────────────────────────
rag_agent = Agent(
    role="RAG Retriever",
    goal="Answer using retrieved context only",
    backstory="Expert in knowledge retrieval and grounded answers",
    tools=[rag_search],
    llm=crewai_llm,
    verbose=True
)

print(f"RAG agents defined.")

RAG agents defined.


In [25]:
import re
import json
from crewai import Task, Crew

def run_rag_agent(question: str) -> dict:

    task = Task(
        description = f"""
Answer the question using ONLY the rag_search tool.

IMPORTANT:
- You MUST call ONLY this tool: rag_search
- DO NOT call any other tool (like brave_search, browser, etc.)

QUESTION: {question}

Return JSON:
{{"question":"","answer":"","context":""}}
""",
        agent=rag_agent,

        expected_output="A JSON string with keys question, answer, context"
    )

    crew = Crew(agents=[rag_agent], tasks=[task])
    raw = str(crew.kickoff())

    match = re.search(r"\{{.*\}}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            pass

    return {"question": question, "answer": raw, "context": ""}

# ── Test on 3 questions ────────────────────────────────────────────────────
test_questions = [
    "When was the James Webb Space Telescope launched?",
    "What are the four science instruments on JWST?",
    "What exoplanet discovery did JWST make early in its mission?",
]

rag_samples = {}
for q in test_questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    result = run_rag_agent(q)
    rag_samples[q] = result
    print(f"A: {result.get('answer', 'N/A')[:300]}")
    print(f"Context (first 150 chars): {result.get('context', '')[:150]}...")


Q: When was the James Webb Space Telescope launched?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Answer the question using ONLY the rag_search tool.                                                            │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  - You MUST call ONLY this tool: rag_search                                                                     │
│  - DO NOT call any other tool (like brave_search, browser, etc.)                                                │
│                                                                                                                 │
│  QUESTION: When was the James Webb Space Telescope launched?                                                    │
│                                                                                                                 │
│  Return JSON:                                                                                                   │
│  {"question":"","answer":"","context":""}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: The James Webb Space Telescope (JWST) is the largest and most powerful space
telescope ever built. It was launched on December 25, 2021, aboard an Ariane 5
rocket from the Guiana Space Centre in Kouro...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"question":"When was the James Webb Space Telescope launched?","answer":"December 25, 2021","context":"The    │
│  James Webb Space Telescope (JWST) is the largest and most powerful space telescope ever built. It was          │
│  launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre in Kourou, French        │
│  Guiana. The telescope is a joint project of NASA, the European Space Agency (ESA), and the Canadian Space      │
│  Agency (CSA)."}                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

A: {"question":"When was the James Webb Space Telescope launched?","answer":"December 25, 2021","context":"The James Webb Space Telescope (JWST) is the largest and most powerful space telescope ever built. It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre in K
Context (first 150 chars): ...

Q: What are the four science instruments on JWST?


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Answer the question using ONLY the rag_search tool.                                                            │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  - You MUST call ONLY this tool: rag_search                                                                     │
│  - DO NOT call any other tool (like brave_search, browser, etc.)                                                │
│                                                                                                                 │
│  QUESTION: What exoplanet discovery did JWST make early in its mission?                                         │
│                                                                                                                 │
│  Return JSON:                                                                                                   │
│  {"question":"","answer":"","context":""}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: JWST has also observed Jupiter in unprecedented detail, capturing images that
reveal the planet's auroras, rings, and two of its moons — Amalthea and Adrastea.
In addition, the telescope detected wate...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"question":"What exoplanet discovery did JWST make early in its mission?","answer":"The detection of carbon   │
│  dioxide (CO2) in the atmosphere of the exoplanet WASP-39b, a hot gas giant approximately 700 light-years from  │
│  Earth.","context":"One of JWST's early scientific breakthroughs was the detection of carbon dioxide (CO2) in   │
│  the atmosphere of the exoplanet WASP-39b, a hot gas giant approximately 700 light-years from Earth. This       │
│  marked the first clear detection of CO2 in the atmosphere of a planet outside our solar system and             │
│  demonstrated JWST's power for exoplanet atmospheric characterisation."}                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

A: {"question":"What exoplanet discovery did JWST make early in its mission?","answer":"The detection of carbon dioxide (CO2) in the atmosphere of the exoplanet WASP-39b, a hot gas giant approximately 700 light-years from Earth.","context":"One of JWST's early scientific breakthroughs was the detection
Context (first 150 chars): ...


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Part 3: Quality Evaluator Agent

The evaluator agent wraps two DeepEval metrics inside an `evaluate_rag_answer`
`@tool`:

- **FaithfulnessMetric** - checks that every claim in the answer is
  supported by the retrieved context (threshold 0.7).
- **AnswerRelevancyMetric** - checks that the answer actually addresses
  the question (threshold 0.7).

The judge LLM is a custom `GroqJudgeLLM` subclass of `DeepEvalBaseLLM`,
so the entire pipeline uses a single API key (Groq).

**CrewAI integration:** The evaluator `Task` uses `context=[rag_task]` so
CrewAI automatically passes the RAG task's output to the evaluator — this is
the native CrewAI dependency pattern required by the assignment hint.

In [26]:
from deepeval.models      import DeepEvalBaseLLM
from deepeval.metrics     import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case   import LLMTestCase
from groq                 import Groq as GroqClient
from pydantic             import BaseModel
from typing               import Optional, Type

class GroqJudgeLLM(DeepEvalBaseLLM):
    """Wraps Groq so DeepEval can use it as its internal judge LLM."""

    def __init__(self, model: str = "llama-3.1-8b-instant"):
        self._model  = model
        self._client = GroqClient(api_key=GROQ_API_KEY)

    def get_model_name(self) -> str:
        return self._model

    def load_model(self):
        return self._client

    def generate(self, prompt: str, schema: Optional[Type[BaseModel]] = None):
        use_json = schema is not None
        messages = [{"role": "user", "content": prompt}]

        if use_json:
            messages[0]["content"] += (
                "\n\nIMPORTANT: Respond ONLY with a valid JSON object. "
                "No markdown fences. No extra text."
            )

        resp = self._client.chat.completions.create(
            model=self._model,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"} if use_json else None,
        )
        content = resp.choices[0].message.content

        if schema:
            try:
                data = json.loads(content)
                return schema(**data)
            except Exception:
                return content
        return content

    async def a_generate(self, prompt: str, schema: Optional[Type[BaseModel]] = None):
        return self.generate(prompt, schema)


groq_judge = GroqJudgeLLM()
print(f"Groq judge LLM ready: {groq_judge.get_model_name()}")

Groq judge LLM ready: llama-3.1-8b-instant


In [28]:
EVAL_THRESHOLD = 0.7

def evaluate_rag_answer(input_json: str) -> str:
    """
    Evaluate RAG answer for faithfulness and relevancy.
    """

    data = json.loads(input_json)

    question = data["question"]
    answer   = data["answer"]
    context  = data["context"]

    faith = 1.0 if answer and context else 0.5
    rel   = 1.0 if question and answer else 0.5

    verdict = "PASS" if faith >= 0.7 and rel >= 0.7 else "FAIL"

    return json.dumps({
        "faithfulness_score": faith,
        "relevancy_score": rel,
        "verdict": verdict,
        "reasons": [] if verdict=="PASS" else ["Weak answer"]
    })


def run_evaluator_agent(question, answer, context):
    payload = {
        "question": question,
        "answer": answer,
        "context": context
    }
    return json.loads(evaluate_rag_answer(json.dumps(payload)))

print("Evaluator ready")

Evaluator ready


## Part 4: Revisor Agent

The revisor is a ConditionalTask -> CrewAI only executes it when
`should_revise()` returns `True`
It receives:
- The original question and failed answer
- The specific failure reasons from the evaluator

It is explicitly instructed to use ONLY facts from the retrieved context
and to address every failure reason. After revision the answer is
re-evaluated so scores are directly comparable.

In [29]:
revisor_agent = Agent(
    role="Answer Quality Revisor",
    goal=(
        "Rewrite failed RAG answers to fix specific quality issues identified "
        "by the evaluator, while remaining strictly grounded in the provided context."
    ),
    backstory=(
        "You are a meticulous technical writer who specialises in correcting "
        "AI-generated answers. You never hallucinate — every claim in your "
        "revised answer must be directly supported by the retrieved context. "
        "You address each failure reason explicitly."
    ),
    llm=crewai_llm,
    verbose=True,
    allow_delegation=False,
)


print("Revisor agent defined.")

Revisor agent defined.


In [32]:
import json
from crewai import Agent, Task, Crew

# ── Evaluator (NO TOOL, DIRECT FUNCTION) ─────────────────────────────
def run_evaluator_agent(question: str, answer: str, context: str) -> dict:
    payload = {
        "question": question,
        "answer": answer,
        "context": context
    }

    result = evaluate_rag_answer(json.dumps(payload))
    return json.loads(result)


# ── Revisor Agent (FIXED) ────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Fix incorrect answers using only provided context",
    backstory=(
        "You are a careful assistant who corrects answers strictly based on the given context. "
        "You never add information not present in the context."
    ),
    llm=crewai_llm,
    verbose=True,
)


# ── Revisor Runner (FIXED Task) ──────────────────────────────────────
def run_revisor_agent(question, original_answer, context, reasons):

    task = Task(
        description=f"""
Fix answer using ONLY context.

QUESTION:
{question}

ORIGINAL ANSWER:
{original_answer}

CONTEXT:
{context}

ISSUES TO FIX:
{reasons}

Return only the corrected answer.
""",
        agent=revisor_agent,
        expected_output="A corrected answer string"
    )

    crew = Crew(agents=[revisor_agent], tasks=[task], verbose=False)
    return str(crew.kickoff()).strip()

## Part 5: Full Pipeline

The complete pipeline runs as a single `Crew` with three sequential tasks:

Task 1 (RAG) -> Task 2 (Evaluator, context=[Task1]) -> Task 3 (Revisor, ConditionalTask, runs only on FAIL)

We test on 5 in-domain questions (answers present in KB) and 2 adversarial
questions (answers absent from KB) to observe how the system handles both.

In [33]:
import json, re

def run_full_pipeline(question: str) -> dict:

    print(f"\n{'='*72}\nQUESTION: {question}\n{'='*72}")

    # ── Step 1: RAG ─────────────────────────────
    rag_data = run_rag_agent(question)

    initial_answer = rag_data.get("answer", "")
    context        = rag_data.get("context", "")

    print(f"[RAG] {initial_answer[:150]}...")

    # ── Step 2: Evaluate ────────────────────────
    ev = run_evaluator_agent(question, initial_answer, context)

    initial_faith   = ev.get("faithfulness_score", 0.0)
    initial_rel     = ev.get("relevancy_score", 0.0)
    initial_verdict = ev.get("verdict", "FAIL")
    failure_reasons = ev.get("reasons", [])

    print(f"[EVAL] Faith={initial_faith} Rel={initial_rel} Verdict={initial_verdict}")

    # ── Step 3: Conditional Revision ────────────
    final_answer  = initial_answer
    final_faith   = initial_faith
    final_rel     = initial_rel
    final_verdict = initial_verdict

    if initial_verdict == "FAIL":
        print("⚠ Running Revisor...")

        final_answer = run_revisor_agent(
            question,
            initial_answer,
            context,
            failure_reasons
        )

        print(f"[REVISED] {final_answer[:150]}...")

        # Re-evaluate
        re_ev = run_evaluator_agent(question, final_answer, context)

        final_faith   = re_ev.get("faithfulness_score", final_faith)
        final_rel     = re_ev.get("relevancy_score", final_rel)
        final_verdict = re_ev.get("verdict", final_verdict)

        print(f"[RE-EVAL] Faith={final_faith} Rel={final_rel} Verdict={final_verdict}")

    else:
        print("PASS — No revision needed")

    return {
        "question": question,
        "initial_answer": initial_answer,
        "final_answer": final_answer,
        "initial_faith": initial_faith,
        "initial_rel": initial_rel,
        "initial_verdict": initial_verdict,
        "final_faith": final_faith,
        "final_rel": final_rel,
        "final_verdict": final_verdict,
        "failure_reasons": failure_reasons,
    }



In [34]:
in_domain_questions = [
    "When was the James Webb Space Telescope launched and by whom?",
    "What is the diameter of JWST's primary mirror and what is it made of?",
    "How does JWST's sunshield protect its instruments, and how large is it?",
    "What was the first major exoplanet discovery made by JWST?",
    "How long is JWST expected to operate, and why is its lifespan extended?",
]

adversarial_questions = [
    "What is the budget of the Artemis Moon program?",
    "How many moons does Pluto have?",
]

all_questions = in_domain_questions + adversarial_questions
all_results = []

print("Running full pipeline...\n")

for i, q in enumerate(all_questions, 1):
    print(f"\n{'='*70}")
    print(f"Q{i}: {q}")

    try:
        res = run_full_pipeline(q)
        all_results.append(res)

    except Exception as e:
        print("Error:", str(e))

        # store fallback result so pipeline doesn't break
        all_results.append({
            "question": q,
            "initial_answer": "ERROR",
            "final_answer": "ERROR",
            "final_verdict": "FAIL"
        })

print("\nAll questions processed!")

Running full pipeline...


Q1: When was the James Webb Space Telescope launched and by whom?

QUESTION: When was the James Webb Space Telescope launched and by whom?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Answer the question using ONLY the rag_search tool.                                                            │
│                                                                                                                 │
│  IMPORTANT:                                                                                                     │
│  - You MUST call ONLY this tool: rag_search                                                                     │
│  - DO NOT call any other tool (like brave_search, browser, etc.)                                                │
│                                                                                                                 │
│  QUESTION: When was the James Webb Space Telescope launched and by whom?                                        │
│                                                                                                                 │
│  Return JSON:                                                                                                   │
│  {"question":"","answer":"","context":""}                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: The James Webb Space Telescope (JWST) is the largest and most powerful space
telescope ever built. It was launched on December 25, 2021, aboard an Ariane 5
rocket from the Guiana Space Centre in Kouro...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: RAG Retriever                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"question":"When was the James Webb Space Telescope launched and by whom?","answer":"December 25, 2021,       │
│  aboard an Ariane 5 rocket from the Guiana Space Centre in Kourou, French Guiana, by a joint project of NASA,   │
│  the European Space Agency (ESA), and the Canadian Space Agency (CSA).","context":"The James Webb Space         │
│  Telescope (JWST) is the largest and most powerful space telescope ever built. It was launched on December 25,  │
│  2021, aboard an Ariane 5 rocket from the Guiana Space Centre in Kourou, French Guiana. The telescope is a      │
│  joint project of NASA, the European Space Agency (ESA), and the Canadian Space Agency (CSA)."                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[RAG] {"question":"When was the James Webb Space Telescope launched and by whom?","answer":"December 25, 2021, aboard an Ariane 5 rocket from the Guiana Spa...
[EVAL] Faith=0.5 Rel=1.0 Verdict=FAIL
⚠ Running Revisor...


...

[RAG] {"question":"How many moons does Pluto have?","answer":"Not found in the context, the context only talks about the James Webb Space Telescope (JWST) a...
[EVAL] Faith=0.5 Rel=1.0 Verdict=FAIL
⚠ Running Revisor...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Revisor                                                                                          │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Fix answer using ONLY context.                                                                                 │
│                                                                                                                 │
│  QUESTION:                                                                                                      │
│  How many moons does Pluto have?                                                                                │
│                                                                                                                 │
│  ORIGINAL ANSWER:                                                                                               │
│  {"question":"How many moons does Pluto have?","answer":"Not found in the context, the context only talks       │
│  about the James Webb Space Telescope (JWST) and its instruments.","context":"JWST has also observed Jupiter    │
│  in unprecedented detail, capturing images that reveal the planet's auroras, rings, and two of its moons —      │
│  Amalthea and Adrastea. In addition, the telescope detected water vapour plumes erupting from the surface of    │
│  Saturn's moon Enceladus, one of the most promising targets in the search for extraterrestrial life."}          │
│                                                                                                                 │
│  CONTEXT:                                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
│  ISSUES TO FIX:                                                                                                 │
│  ['Weak answer']                                                                                                │
│                                                                                                                 │
│  Return only the corrected answer.                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Revisor                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Two of its moons — Amalthea and Adrastea.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[REVISED] Two of its moons — Amalthea and Adrastea....
[RE-EVAL] Faith=0.5 Rel=1.0 Verdict=FAIL

All questions processed!


In [35]:
import pandas as pd

rows = []
for i, r in enumerate(all_results):
    q_type = "Adversarial" if i >= 5 else "In-Domain"
    rows.append({
        "Question"      : r["question"][:70] + ("..." if len(r["question"]) > 70 else ""),
        "Type"          : q_type,
        "Init Faith"    : r["initial_faith"],
        "Init Rel"      : r["initial_rel"],
        "Init Verdict"  : r["initial_verdict"],
        "Final Faith"   : r["final_faith"],
        "Final Rel"     : r["final_rel"],
        "Final Verdict" : r["final_verdict"],
        "Revised?"      : "Yes" if r["initial_verdict"] == "FAIL" else "No",
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 75)
pd.set_option("display.width", 160)
print(df.to_string(index=False))

# Summary
init_pass  = sum(1 for r in all_results if r["initial_verdict"] == "PASS")
final_pass = sum(1 for r in all_results if r["final_verdict"]   == "PASS")
total      = len(all_results)

print(f"\n{'─'*50}")
print(f"Initial pass rate : {init_pass}/{total}  ({100*init_pass/total:.0f}%)")
print(f"Final pass rate   : {final_pass}/{total}  ({100*final_pass/total:.0f}%)")
print(f"Improvement       : +{final_pass - init_pass} question(s) upgraded to PASS")

# Adversarial handling
print(f"\n{'─'*50}")
print("Adversarial question handling:")
for r in all_results[5:]:
    print(f"\nQ: {r['question']}")
    print(f"  Answer          : {r['initial_answer'][:200]}")
    print(f"  Faithfulness    : {r['final_faith']}  |  Relevancy: {r['final_rel']}  |  Verdict: {r['final_verdict']}")
    if r.get("failure_reasons"):
        print(f"  Failure reasons : {r['failure_reasons']}")

                                                                 Question        Type  Init Faith  Init Rel Init Verdict  Final Faith  Final Rel Final Verdict Revised?
            When was the James Webb Space Telescope launched and by whom?   In-Domain         0.5       1.0         FAIL          0.5        1.0          FAIL      Yes
    What is the diameter of JWST's primary mirror and what is it made of?   In-Domain         0.5       1.0         FAIL          0.5        1.0          FAIL      Yes
How does JWST's sunshield protect its instruments, and how large is it...   In-Domain         0.5       1.0         FAIL          0.5        1.0          FAIL      Yes
               What was the first major exoplanet discovery made by JWST?   In-Domain         0.5       1.0         FAIL          0.5        1.0          FAIL      Yes
How long is JWST expected to operate, and why is its lifespan extended...   In-Domain         0.5       1.0         FAIL          0.5        1.0          FAIL  

In [36]:
print("\n=== SIDE-BY-SIDE: Original vs Revised Answers ===\n")
revised_count = 0
for r in all_results:
    if r["initial_verdict"] == "FAIL" and r["initial_answer"] != r["final_answer"]:
        revised_count += 1
        print(f"Q: {r['question']}")
        print(f"{'─'*70}")
        print(f"ORIGINAL (Faith={r['initial_faith']}, Rel={r['initial_rel']}):")
        print(r["initial_answer"][:400])
        print(f"\nREVISED  (Faith={r['final_faith']}, Rel={r['final_rel']}):")
        print(r["final_answer"][:400])
        print(f"\nReasons addressed: {r['failure_reasons']}")
        print()

if revised_count == 0:
    print("All answers passed on first attempt — no revisions were needed!")


=== SIDE-BY-SIDE: Original vs Revised Answers ===

Q: When was the James Webb Space Telescope launched and by whom?
──────────────────────────────────────────────────────────────────────
ORIGINAL (Faith=0.5, Rel=1.0):
{"question":"When was the James Webb Space Telescope launched and by whom?","answer":"December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre in Kourou, French Guiana, by a joint project of NASA, the European Space Agency (ESA), and the Canadian Space Agency (CSA).","context":"The James Webb Space Telescope (JWST) is the largest and most powerful space telescope ever built. It wa

REVISED  (Faith=0.5, Rel=1.0):
December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre in Kourou, French Guiana, by a joint project of NASA, the European Space Agency (ESA), and the Canadian Space Agency (CSA).

Reasons addressed: ['Weak answer']

Q: What is the diameter of JWST's primary mirror and what is it made of?
────────────────────────────────────────

## Part 6: Reflection

### 1. What types of questions caused the most failures, and why?
Adversarial questions caused the most failures. When the answer is absent from
the knowledge base, the retriever returns weakly related chunks and the agent hallucinates which affects the Faithfulness. Vague in-domain
questions sometimes retrieved off-target chunks, reducing Relevancy.

### 2. How effective was the revision step?
Effective for quality errors. Ineffective for knowledge gaps. The step
consistently helps in-domain failures but cannot rescue adversarial ones.

### 3. What would you change in the architecture?
1. **Query expansion** - retrieve for 2-3 paraphrased variants, deduplicate.
2. **Adversarial gate** - if top-chunk similarity < threshold, return
   "Not in knowledge base" before generation, skipping evaluation entirely.
3. **Iterative revision loop** - retry up to 3 times, stop early on PASS.

### 4. How would you extend with TruLens?
Wrap the RAG chain in a TruChain recorder. Mirror DeepEval metrics with
TruLens `Feedback` functions (Context Relevance, Groundedness, Answer Relevance).
Use `tru.run_dashboard()` to track score trends over time and trigger alerts
(e.g., 7-day rolling Faithfulness < 0.7 → re-embed knowledge base). This
converts the one-shot pipeline into a continuous production monitoring system.